In [1]:
from __future__ import division
from __future__ import print_function
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.18.1
#   kernelspec:
#     display_name: Python 2
#     language: python
#     name: python2
# ---

import warnings
warnings.filterwarnings("ignore")

<h1>Miscellanous Experiments on the Cohort</h1>
- Autopsy Rates
- Table One
- Pairwise Correlations Between Mistrust Scores and OASIS
- Sentiment Disparities, Stratified by Mistrust OR Race

In [2]:
from __future__ import absolute_import
from future import standard_library
from six.moves import range
standard_library.install_aliases()
from builtins import range
from past.utils import old_div
import pickle as pickle
import numpy as np
import pandas as pd
import psycopg2
from time import strftime, gmtime
import tqdm

In [3]:
# create a database connection
sqluser = 'wboag'
dbname = 'mimiciv'

# Connect to the database
# con = psycopg2.connect(dbname=dbname, user=sqluser, host="/var/run/postgresql")
con = psycopg2.connect(dbname=dbname, user=sqluser, host="host.docker.internal", port="5432")

<h1>Load Data</h1>

In [4]:
def normalize_race(race):
    if 'HISPANIC' in race:
        return 'Hispanic'
    if 'SOUTH AMERICAN' in race:
        return 'Hispanic'
    if 'AMERICAN INDIAN' in race:
        return 'Native American'
    if 'ASIAN' in race:
        return 'Asian'
    if 'BLACK' in race:
        return 'Black'
    if 'UNKNOWN/NOT SPECIFIED' in race:
        return 'Not Specified'
    if 'WHITE' in race:
        return 'White'
    #print race
    return 'Other'

def normalize_insurance(ins):
    if ins in ['Government', 'Medicaid', 'Medicare']:
        return 'Public'
    elif ins == 'Private':
        return 'Private'
    else:
        return 'Self-Pay'
    
def normalize_discharge(disch):
    if not disch or pd.isna(disch):
        return 'other'
    if disch.startswith('HOSPICE'):
        return 'Hospice'
    if disch == 'DEAD/EXPIRED':
        return 'Deceased'
    if disch.startswith('SNF'):
        return 'Skilled Nursing Facility'
    return 'other'

def normalize_age(age):
    return min(age, 90)

In [5]:
# [repro] note: updated age to admission_age
demographics_query = """
SELECT DISTINCT
    subject_id,
    hadm_id,
    gender,
    CASE
      WHEN admission_age > 89 THEN 90
      ELSE ROUND(admission_age)
    END AS age,
    race AS ethnicity
FROM mimiciv_derived.icustay_detail
WHERE first_icu_stay = TRUE;
"""
demographics = pd.read_sql_query(demographics_query, con)
demographics.head()

,subject_id,hadm_id,gender,age,ethnicity
0,15032560,25673375,F,60.0,WHITE
1,14276582,26997248,F,51.0,PATIENT DECLINED TO ANSWER
2,10860177,24427969,F,87.0,WHITE
3,10117273,25087476,M,74.0,WHITE
4,12161031,24510167,M,74.0,WHITE


In [6]:
# admissions info
print(strftime("%Y-%m-%d %H:%M:%s"))
discharge_query = """
SELECT DISTINCT subject_id,hadm_id,race AS ethnicity,insurance,discharge_location,admittime,dischtime
FROM mimiciv_hosp.admissions
WHERE discharge_location IS NOT NULL;
"""
discharge = pd.read_sql_query(discharge_query, con)

discharge['discharge_location'] = discharge['discharge_location'].apply(normalize_discharge)
print(strftime("%Y-%m-%d %H:%M:%s"))

2025-12-04 03:50:1764820236
2025-12-04 03:50:1764820237


In [7]:
# EOL Cohort

print(strftime("%Y-%m-%d %H:%M:%s"))

# patients who died or went to hospice
#eol_locations = {'Hospice', 'Deceased'}
eol_locations = {'Hospice', 'Deceased', 'Skilled Nursing Facility'}
disch = discharge.loc[discharge['discharge_location'].isin(eol_locations)]

ids =  set(disch.hadm_id.values)
eol_cohort_initial = discharge.loc[discharge.hadm_id.isin(ids)]

inds_at_least_6hrs = eol_cohort_initial['dischtime'] - eol_cohort_initial['admittime'] > pd.Timedelta(days=1)
eol_cohort_initial = eol_cohort_initial.loc[inds_at_least_6hrs]


# add demographics info
eol_cohort = pd.merge(eol_cohort_initial, demographics, on=['hadm_id','ethnicity'])
eol_cohort = eol_cohort.rename(columns={'ethnicity':'race'})

# normalize columns of data
eol_cohort['race'              ] = eol_cohort['race'              ].apply(normalize_race)
eol_cohort['insurance'         ] = eol_cohort['insurance'         ].apply(normalize_insurance)
eol_cohort['age'               ] = eol_cohort['age'               ].apply(normalize_age)

los = eol_cohort['dischtime'] - eol_cohort['admittime']
eol_cohort['los'] = los.apply(lambda t:t.seconds/3600.)

# make sure each hadm_id has only died once
assert len(eol_cohort) == len(set(eol_cohort['hadm_id'].values))
print('eol subjects:', len(set(eol_cohort['hadm_id'].values)))

print(strftime("%Y-%m-%d %H:%M:%s"))

#eol_cohort.head()

2025-12-04 03:50:1764820237
eol subjects: 2081
2025-12-04 03:50:1764820237


<h1>Autopsy Rates</h1>

In [8]:
# LABEL: autopsy consent/decline

# Query mimic for notes
# MIMIC-IV equivalent of MIMIC-III noteevents query
notes_query = """
SELECT DISTINCT
    d.hadm_id,
    'Discharge summary' AS category,
    d.text,
    d.charttime::date AS chartdate,
    d.charttime
FROM mimiciv_note.discharge AS d
WHERE d.hadm_id IS NOT NULL

UNION ALL

SELECT DISTINCT
    r.hadm_id,
    'Radiology report' AS category,
    r.text,
    r.charttime::date AS chartdate,
    r.charttime
FROM mimiciv_note.radiology AS r
WHERE r.hadm_id IS NOT NULL;
"""
notes = pd.read_sql_query(notes_query, con)

autopsy_consent = []
autopsy_decline = []
for hadm_id,rows in tqdm.tqdm(notes.groupby('hadm_id')):
    consented = False
    declined = False
    for text in rows.text.values:
        for line in text.lower().split('\n'):
            if 'autopsy' in line:
                if 'decline' in line:
                    declined = True
                if 'not consent' in line:
                    declined = True
                if 'refuse' in line:
                    declined = True
                if 'denied' in line:
                    declined = True
                    
                if 'consent' in line:
                    consented = True
                if 'agree' in line:
                    consented = True
                if 'request' in line:
                    consented = True

    # probably some "declined donation but consented to autopsy" or something confusing. just ignore hard cases
    if consented and declined:
        continue

    if consented:
        autopsy_consent.append(hadm_id)
    if declined:
        autopsy_decline.append(hadm_id)

100%|██████████| 374285/374285 [00:24<00:00, 15061.76it/s]


In [9]:

for race in ['White', 'Black', 'Asian', 'Native American', 'Hispanic', 'Not Specified', 'Other']:
    cohort = eol_cohort.loc[eol_cohort['race']==race]
    consent = cohort.loc[cohort['hadm_id'].isin(autopsy_consent)]
    decline = cohort.loc[cohort['hadm_id'].isin(autopsy_decline)]

    print(race)
    print('\tautopsy consent:', len(consent))
    print('\tautopsy decline:', len(decline))
    print('\t% consent:', old_div(len(consent),(len(consent)+len(decline)+1e-9)))
    print() 

White
	autopsy consent: 2
	autopsy decline: 1
	% consent: 0.6666666664444444

Black
	autopsy consent: 0
	autopsy decline: 0
	% consent: 0.0

Asian
	autopsy consent: 0
	autopsy decline: 0
	% consent: 0.0

Native American
	autopsy consent: 0
	autopsy decline: 0
	% consent: 0.0

Hispanic
	autopsy consent: 0
	autopsy decline: 0
	% consent: 0.0

Not Specified
	autopsy consent: 0
	autopsy decline: 0
	% consent: 0.0

Other
	autopsy consent: 0
	autopsy decline: 0
	% consent: 0.0



<h1>Table One</h1>

In [10]:
# Table One

from tableone import TableOne
import pandas as pd
import matplotlib.pyplot as plt

los = eol_cohort['dischtime'] - eol_cohort['admittime']
eol_cohort['los'] = los.apply(lambda t:t.seconds/3600.)

# optionally, a categorical variable for stratification
groupby = ['race']

# columns to be summarized
columns = ['discharge_location', 'gender', 'los', 'age'] 

# columns containing categorical variables
categorical = ['discharge_location', 'gender']

# non-normal variables
nonnormal = ['age', 'los']

# alternative labels
labels={'los': 'Length of stay', 'age': 'Age', 'race':'Race',
        'gender':'Gender', 'discharge_location':'Discharge Location'}

# combine all information
#grouped_df = pd.merge(eol_cohort, demographics, on=['hadm_id'])
grouped_df = eol_cohort

# create an instance of TableOne with the input arguments
grouped_table = TableOne(grouped_df, columns, categorical, groupby, nonnormal, labels=labels, isnull=False, pval=True)

# view tableone
grouped_table

/usr/local/lib/python3.6/site-packages/tableone/tableone.py:219: DeprecationWarning: The labels argument is deprecated; use rename instead.
  "rename instead.", DeprecationWarning)
/usr/local/lib/python3.6/site-packages/tableone/tableone.py:227: DeprecationWarning: The isnull argument is deprecated; use missing instead.
  "missing instead.", DeprecationWarning)


Grouped by Race                                                                                                                    
                                                 Overall             Asian             Black          Hispanic   Native American             Other             White P-Value
n                                                   2081                76               224                64                 6               266              1445        
Discharge Location, n (%)      Hospice      2081 (100.0)        76 (100.0)       224 (100.0)        64 (100.0)         6 (100.0)       266 (100.0)      1445 (100.0)   1.000
Gender, n (%)                  F             1012 (48.6)         32 (42.1)        124 (55.4)         26 (40.6)          4 (66.7)        143 (53.8)        683 (47.3)   0.040
                               M             1069 (51.4)         44 (57.9)        100 (44.6)         38 (59.4)          2 (33.3)        123 (46.2)        762 (52.7)        
Length of stay, median [Q1,Q3]           15.9 [7.7,19.8]   14.9 [8.9,18.8]   15.3 [6.9,20.3]   16.3 [5.8,19.6]   16.5 [5.6,18.7]   16.7 [9.0,20.0]   15.7 [7.7,19.8]   0.720
Age, median [Q1,Q3]                     75.0 [64.0,85.0]  71.0 [60.0,82.2]  75.0 [63.0,83.2]  67.0 [55.0,81.0]  77.0 [70.2,84.5]  78.0 [66.0,85.0]  75.0 [64.0,85.0]   0.003
[1] Chi-squared tests for the following variables may be invalid due to the low number of observations: gender.

<h1>All Metrics Correlation</h1>

In [11]:
# Load all scores

def normalize(scores):
    mu  = sum(scores.values())
    std = np.std(list(scores.values()))
    return {k:old_div((v-mu),std) for k,v in list(scores.items())}

# Get the OASIS scores
oasis_query = 'SELECT distinct hadm_id,max(oasis) as oasis FROM mimiciv_derived.oasis GROUP BY hadm_id'
oasis = pd.read_sql_query(oasis_query, con)
oasis_scores = normalize(dict(oasis[['hadm_id','oasis']].values))

# Mistrust scores
with open('../data/mistrust_noncompliant.pkl', 'rb') as f:
    noncompliant_scores = normalize(pickle.load(f))
with open('../data/mistrust_autopsy.pkl', 'rb') as f:
    autopsy_scores = normalize(pickle.load(f))
with open('../data/neg_sentiment.pkl', 'rb') as f:
    negsent_scores = normalize(pickle.load(f))

# To make my for loop work easier
race_scores = {}
for i,row in eol_cohort.iterrows():
    if row.race == 'White':
        race_scores[row.hadm_id] = 0
    elif row.race == 'Black':
        race_scores[row.hadm_id] = 1

sa_ids = set(negsent_scores.keys()) & set(oasis_scores.keys()) & \
         set(noncompliant_scores.keys()) & set(autopsy_scores.keys()) & set(race_scores.keys())

print(len(sa_ids))

1054


In [12]:
def select(scores):
    return [scores[hadm_id] for hadm_id in sa_ids]
    
all_scores = {'oasis':select(oasis_scores), 'sentiment':select(negsent_scores), 
              'noncompliant':select(noncompliant_scores), 'autopsy':select(autopsy_scores)}
metrics = ['oasis', 'noncompliant', 'autopsy', 'sentiment']

from scipy.stats import pearsonr

print(' '*15, end=' ')
for m in metrics:
    print('%19s' % m, end=' ')
print()
for i in range(len(metrics)):
    m1 = metrics[i]
    print('%-15s' % m1, end=' ') 
    for j in range(len(metrics)):
        m2 = metrics[j]
        print('%19.3f' % pearsonr(all_scores[m1], all_scores[m2])[0], end=' ') 
    print()

                              oasis        noncompliant             autopsy           sentiment 
oasis                         1.000              -0.015              -0.094               0.086 
noncompliant                 -0.015               1.000               0.380               0.037 
autopsy                      -0.094               0.380               1.000               0.032 
sentiment                     0.086               0.037               0.032               1.000 


<h1>Trust-based Sentiment Disparities</h1>

In [13]:

from scipy.stats import mannwhitneyu

def significant_mean_diff_test(label, white, black):
    W = list(white.values())
    B = list(black.values())
    stat, pval = mannwhitneyu(W, B)

    n1 = len(W)
    n2 = len(B)
    
    mW = sorted(W)[old_div(n1,2)]
    mB = sorted(B)[old_div(n2,2)]
    
    #print '%-15s: p=%f' % (label,pval)
    print('%-15s: n1=%d median(p1)=%.3f n2=%d median(p2)=%.3f p=%.5f' % (label,n1,mW,n2,mB,pval))
    

score_names = {'race':race_scores,
               'oasis':oasis_scores,
               'noncompliant':noncompliant_scores,
               'autopsy':autopsy_scores}

num_black = sum([race_scores[hadm_id] for hadm_id in sa_ids])

for name,scores in list(score_names.items()):
    sscores = sorted([(hadm_id,scores[hadm_id]) for hadm_id in sa_ids], key=lambda t:t[1])
    low_scores  = dict(sscores[:-num_black])
    high_scores = dict(sscores[ -num_black:])

    # Stratify by trust
    low  = {hadm_id:negsent_scores[hadm_id] for hadm_id in  list(low_scores.keys())}
    high = {hadm_id:negsent_scores[hadm_id] for hadm_id in list(high_scores.keys())}
    significant_mean_diff_test(name, low, high)

race           : n1=920 median(p1)=0.264 n2=134 median(p2)=0.410 p=0.22815
oasis          : n1=920 median(p1)=0.264 n2=134 median(p2)=0.335 p=0.03846
noncompliant   : n1=920 median(p1)=0.271 n2=134 median(p2)=0.254 p=0.23575
autopsy        : n1=920 median(p1)=0.262 n2=134 median(p2)=0.330 p=0.20241


/usr/local/lib/python3.6/site-packages/scipy/stats/stats.py:7002: DeprecationWarning: Calling `mannwhitneyu` without specifying `alternative` is deprecated.
  "`alternative` is deprecated.", DeprecationWarning)
/usr/local/lib/python3.6/site-packages/scipy/stats/stats.py:7002: DeprecationWarning: Calling `mannwhitneyu` without specifying `alternative` is deprecated.
  "`alternative` is deprecated.", DeprecationWarning)
/usr/local/lib/python3.6/site-packages/scipy/stats/stats.py:7002: DeprecationWarning: Calling `mannwhitneyu` without specifying `alternative` is deprecated.
  "`alternative` is deprecated.", DeprecationWarning)
/usr/local/lib/python3.6/site-packages/scipy/stats/stats.py:7002: DeprecationWarning: Calling `mannwhitneyu` without specifying `alternative` is deprecated.
  "`alternative` is deprecated.", DeprecationWarning)
